In [1]:
!pwd

/home/gsemonis/opencvCourses/opencv_2/project4


In [2]:
%%bash
if [ ! -d "darknet" ]; then
    echo "darknet directory not found. Cloning..."
    git clone https://github.com/AlexeyAB/darknet.git
else
    echo "darknet directory already exists."
fi


darknet directory not found. Cloning...


Cloning into 'darknet'...


In [3]:
#change to darknet dir
%cd darknet

/home/gsemonis/opencvCourses/opencv_2/project4/darknet


In [5]:
#copy editied make file. i have a 4090 so had to edit the arch line, couldn't run with cudnn
!cp ../Makefile ./Makefile
print("Building. . . It might take 2-3 minutes")
!make clean
!make &> build_log.txt

Building. . . It might take 2-3 minutes
rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./obj/go.o ./obj/batchnorm_layer.

In [28]:
%%bash
DATASET_DIR="/home/gsemonis/opencvCourses/opencv_2/project4/dataset"
ZIP_FILE="images.zip"

if [ -d "$DATASET_DIR" ]; then
    echo "Dataset directory already exists: $DATASET_DIR"
    exit 0
fi
mkdir -p "$DATASET_DIR"
wget -O "$ZIP_FILE" "https://www.dropbox.com/s/uq0x32w70c390fb/mask_no-mask_dataset.zip?dl=1"
unzip "$ZIP_FILE" -d "$DATASET_DIR"
rm "$ZIP_FILE"

Dataset directory already exists: /home/gsemonis/opencvCourses/opencv_2/project4/dataset


In [24]:
import random
import os
from pathlib import Path

data_dir = Path('/home/gsemonis/opencvCourses/opencv_2/project4/dataset')
darknet_project_dir = Path('/home/gsemonis/opencvCourses/opencv_2/project4/darknet/project4')
darknet_project_dir.mkdir(parents=True, exist_ok=True)

train_ratio = 0.8
seed = 42

image_extensions = {'.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG'}
all_paths = list(data_dir.glob('*'))
all_file_names = {path.name for path in all_paths}
labeled_images = [
    path for path in all_paths
    if path.suffix in image_extensions and (f'{path.stem}.txt' in all_file_names)
]

print(f'found {len(labeled_images)} images')

random.seed(seed)
random.shuffle(labeled_images)

split_idx = int(len(labeled_images) * train_ratio)
train_images = labeled_images[:split_idx]
test_images  = labeled_images[split_idx:]

print(f'Train: {len(train_images)}')
print(f'Test:  {len(test_images)}')

train_file = darknet_project_dir / "train.txt"
test_file  = darknet_project_dir / "test.txt"

with train_file.open('w') as f:
    for img in train_images:
        f.write(str(img.resolve()) + os.linesep)

with test_file.open('w') as f:
    for img in test_images:
        f.write(str(img.resolve()) + os.linesep)

found 1356 images
Train: 1084
Test:  272


In [29]:
!mkdir -p '/home/gsemonis/opencvCourses/opencv_2/project4/darknet/project4/backup'

In [32]:
names_file = darknet_project_dir / 'names.txt'
data_file = darknet_project_dir / 'yoloV3_mask.data'

with names_file.open('w') as f:
    f.write('mask' +  os.linesep)
    f.write('no_mask' + os.linesep)            

with data_file.open('w') as f:
    f.write('classes = 2' + os.linesep)
    f.write(f'train = {train_file.resolve()}{os.linesep}')
    f.write(f'valid = {test_file.resolve()}{os.linesep}')
    f.write(f'names = {names_file.resolve()}{os.linesep}')
    f.write('backup = /home/gsemonis/opencvCourses/opencv_2/project4/darknet/project4/backup' + os.linesep)

In [37]:
import shutil
new_cfg_path = darknet_project_dir / 'yolov3Masks.cfg'
base_cfg_path = Path('/home/gsemonis/opencvCourses/opencv_2/project4/darknet') / 'cfg/yolov3-voc.cfg'
if not new_cfg_path.is_file():
    shutil.copyfile(base_cfg_path, new_cfg_path)
    print('go modify the cfg file')
else:
    print('cfg exists check it\'s contents')
    

cfg exists check it's contents


In [39]:
!wget -O project4/darknet53.conv.74 "https://www.dropbox.com/s/18dwbfth7prbf0h/darknet53.conv.74?dl=1"




--2025-11-27 01:37:11--  https://www.dropbox.com/s/18dwbfth7prbf0h/darknet53.conv.74?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:601b:18::a27d:812
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/cte4fqetysju1dsccfjco/darknet53.conv.74?rlkey=bluydjd10mna3w539cuyrqf6n&dl=1 [following]
--2025-11-27 01:37:11--  https://www.dropbox.com/scl/fi/cte4fqetysju1dsccfjco/darknet53.conv.74?rlkey=bluydjd10mna3w539cuyrqf6n&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc1ab74aa31ed5d9f88e1b07762e.dl.dropboxusercontent.com/cd/0/inline/C1_EJp2YlMLunmg1dYNVyQ-esGV5wsmueYO8teeEzbHcaT-HHhC7Y9HqSyzzER-B_rOWSTOyTLm82k7z4h9AAE0JntkNOrnXIqtuEwwweyJ3nDFzn8ApALZen5fzdAu_NbwsJc7pnwk-Q6SNnJ-KdpCJ/file?dl=1# [following]
--2025-11-27 01:37:12--  https://uc1ab74aa31ed5d9f88e1b07762e.dl.d

In [ ]:
!./darknet detector train ./project4/yoloV3_mask.data ./project4/yolov3Masks.cfg ./project4/darknet53.conv.74 -dont_show -map 2> ./project4/train_log.txt


In [ ]:
!./darknet detector demo ./project4/yoloV3_mask.data ./project4/yolov3Masks.cfg ./project4/backup/yolov3Masks_best.weights ../test-video1.mp4 -thresh .6 -out_filename ../out-vid1Yolo3.avi -dont_show

In [ ]:
!./darknet detector demo ./project4/yoloV3_mask.data ./project4/yolov3Masks.cfg ./project4/backup/yolov3Masks_best.weights ../test-video2.mp4 -thresh .6 -out_filename ../out-vid2Yolo3.avi -dont_show